# Tool Calling Basics
## Problem Statement
Build a 4-tool agent that routes DevContext AI queries to the right tool using Groq tool calling.

## My Approach
I knew from the problem statement that the real challenge wasn't the API loop - it was making the model route correctly without any if/else on my side. So I spent most of the pre-coding time on tool descriptions rather than code. 

My mental model: descriptions are prompts the model reads to decide which tool fits this query. If two tools sound similar, the model will guess. I tried to give each tool a clear "use this ONLY when..." boundary - especially between search_codebase (passive lookup) and analyze_diff (active impact analysis), since both relate to code but serve completely different intents.

## What I Learned
Tool descriptions are the actual logic of your agent — they're not documentation. The model treats them as decision rules, so every word counts. I learned that "Do NOT use for X — use Y instead" is often more useful than explaining what a tool does, because it handles the ambiguous overlap cases. 

I also learned that json.dumps() on the tool result is non-negotiable — passing a raw Python list breaks silently on some models, and the bug is invisible until you print messages. The routing audit table (expected vs actual) was the most valuable thing I added — without it, you're just printing outputs, not testing routing.

## Where I Got Stuck
The analyze_diff vs search_codebase boundary was the hardest to get right.

My first description for analyze_diff was generic: "Analyze the potential impact of a code change" — which works for obvious queries like "blast radius?" but fails for phrasing like "I changed X — what files are affected?". The model read "what files are affected" as a lookup task and called search_codebase. 

The fix was adding explicit trigger phrases into the description: "use when the user says they ARE CHANGING or HAVE CHANGED a function" — once I matched the user's natural language pattern in the description, routing snapped into place.

## What I'd Do Differently
I'd test each tool description in isolation with 2–3 adversarial queries before running the full suite — it's much faster to iterate on one description at a time than to debug a 5-query run and guess which description caused the failure. 

And next time I'd try tool_choice="required" on the ambiguous queries to see if forcing a tool call changes routing behavior.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [26]:
TEST_QUERIES = [
    ("how does the auth middleware work?",                                          "search_docs"),
    ("what does the users table store?",                                            "search_schema"),
    ("I changed validate_token() to require a scope param — what files are affected?", "analyze_diff"),
    ("explain the payment flow",                                                    "search_docs"),
    ("add_payment_method() now returns a dict instead of bool — blast radius?",     "analyze_diff"),
]

In [27]:
#functions
def search_codebase(query: str) -> list[dict]:
    return [
        {"file": "auth/middleware.py", "snippet": "def validate_token(user_id, token): ..."},
        {"file": "api/routes.py", "snippet": "from auth.middleware import validate_token"},
    ]

def search_docs(query: str) -> list[dict]:
    return [
        {"doc": "AUTH.md", "section": "## Middleware", "content": "The auth middleware validates JWTs on every request..."},
    ]

def search_schema(query: str) -> list[dict]:
    return [
        {"table": "users", "columns": "id, email, created_at, role"},
    ]

def analyze_diff(changed_function: str, new_signature: str) -> list[dict]:
    return [
        {"file": "api/routes.py", "line": 42, "risk": "calls validate_token with old signature"},
        {"file": "tests/test_auth.py", "line": 17, "risk": "mocks validate_token - mock signature outdated"},
    ]

In [28]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"  

In [29]:

# --- Define the tools ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_codebase",
            "description": (
                "Search the indexed codebase using a semantic query."
                "Use this to find relevant files, functions, or code snippets that relate to a specific topic, bug, or pattern. "
                "Returns the top matching code chunks with file paths."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "A specific, targeted search phrase describing the code pattern or concept to find. "
                            "Example: 'users table'"
                        )
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": (
                "Search the indexed document database using a semantic query."
                "Use this to find relevant doc, sections, or content that relate to a specific topic, bug, or pattern. "
                "Returns the top matching doc, sections & content."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "A specific, targeted search phrase describing the code pattern or concept to find. "
                            "Example: 'working of auth middleware'"
                        )
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_schema",
            "description": (
                "Search the indexed schema database using a semantic query."
                "Use this to find relevant tables & column that relate to a specific topic, bug, or pattern. "
                "Returns the top matching table & column details"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "A specific, targeted search phrase describing the concept or tables to find. "
                            "Example: 'database connection pooling' or 'error handling in file upload'"
                        )
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_diff",
            "description": (
                "Analyze the potential impact (blast radius) of a code change. "
                "Use this when a user describes a modification to a function and wants to know what else might break."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "changed_function": {
                        "type": "string",
                        "description": (
                            "A specific function name which needs to be updated & can be found through search_coedbase is present"
                            "Example: 'api/routes.py'"
                        )
                    },
                    "new_signature": {
                        "type": "string",
                        "description": (
                            "A specific phrase describing the change in function"
                            "Example: 'database connection pooling' or 'error handling in file upload'"
                        )
                    }
                },
                "required": ["changed_function","new_signature"],
            },
        },
    }
]

In [30]:
TOOL_MAP = {
    "search_codebase": search_codebase,
    "search_docs": search_docs,
    "search_schema": search_schema,
    "analyze_diff": analyze_diff
}

system_messages = {
        "role": "system",
        "content": f'''
        You are a Q&A and PR impact agent. 
        If user have given a pattern/bug/topic , search it accross all search_ tools ."
        If user have some code changes , analyze difference and answer accordingly.

        Output should always have source present at the end of answer [doc/file/table > section]
        Output example : "The auth middleware validates JWTs on every request [AUTH.md > Middleware]"
        Answer must Be brief in 1 sentence.'''
    }


In [31]:
def run_agent(query: str, max_turns: int = 5) -> dict:
    """
    Run one query through the tool-calling agent.
    Returns a dict with: answer, tools_called (list), raw_messages.
    """
    messages = [system_messages, {"role": "user", "content": query}]
    tools_called = []

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            tools=tools,
            tool_choice="auto",
            messages=messages,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        messages.append(msg)

        if finish == "stop":
            return {"answer": msg.content, "tools_called": tools_called, "messages": messages}

        if finish == "tool_calls" and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)
                tools_called.append(tool_name)

                func = TOOL_MAP.get(tool_name)
                if func is None:
                    result = {"error": f"Tool '{tool_name}' not registered."}
                else:
                    result = func(**tool_args)

                # Tool result MUST be a string — json.dumps even if it's already a list/dict
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result),
                })

    return {"answer": "[max turns exceeded]", "tools_called": tools_called, "messages": messages}

In [ ]:
#Testing toll calling

result=run_agent(TEST_QUERIES[0])

print(f"{'Query':<15}: {TEST_QUERIES[0]}")
actual_tools = result["tools_called"]
answer = result['answer']
print(f"{'Tool called':<15}: {actual_tools}")
print(f"{'Answer':<15}: {answer}")

Query          : how does the auth middleware work?
Tool called    : ['search_docs', 'search_docs']
Answer         : The auth middleware validates JWTs on every request [AUTH.md > Middleware].


In [33]:
routing_results = []  # collect for summary

for query, expected_tool in TEST_QUERIES:
    print(f"{'Query':<15} : {query}")
    print(f"{'Expected Tool':<15} : {expected_tool}")

    result = run_agent(query)
    actual_tools = result["tools_called"]
    first_tool = actual_tools[0] if actual_tools else "[no tool called]"

    routed_correctly = (first_tool == expected_tool)
    status = "✅ CORRECT" if routed_correctly else f"❌ WRONG — called {first_tool}"

    print(f"{'Tool called':<15}: {first_tool}  {status}")
    print(f"{'Answer':<15} : {result['answer']}")
    print("-" * 30)

    routing_results.append({
        "query": query,
        "expected": expected_tool,
        "actual": first_tool,
        "correct": routed_correctly,
        "answer": result["answer"],
    })

Query           : how does the auth middleware work?
Expected Tool   : search_docs
Tool called    : search_docs  ✅ CORRECT
Answer          : The auth middleware validates JWTs on every request [AUTH.md > Middleware].
------------------------------
Query           : what does the users table store?
Expected Tool   : search_schema
Tool called    : search_schema  ✅ CORRECT
Answer          : The users table stores the user id, email, creation timestamp, and role. [users table > schema]
------------------------------
Query           : I changed validate_token() to require a scope param — what files are affected?
Expected Tool   : analyze_diff
Tool called    : analyze_diff  ✅ CORRECT
Answer          : The files affected by this change are api/routes.py and tests/test_auth.py [api/routes.py > lines 42, tests/test_auth.py > line 17].
------------------------------
Query           : explain the payment flow
Expected Tool   : search_docs
Tool called    : search_schema  ❌ WRONG — called search_sc

## Refactored Solution

In [34]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_codebase",
            "description": (
                "Search the indexed source code for implementation details — functions, classes, imports, or logic flow. "
                "Use when the user asks HOW something is implemented in the code (e.g. 'how is payment processed', "
                "'show me where auth is called'). "
                "Do NOT use for explaining concepts (use search_docs) or for database table structure (use search_schema). "
                "Do NOT use when the user is describing a code change and asking what will break — use analyze_diff instead."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Specific function name, module, or pattern to find in source code. E.g. 'validate_token implementation'"
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": (
                "Search onboarding documentation, architecture docs, and markdown guides for conceptual explanations. "
                "Use when the user wants to UNDERSTAND how a system or feature works at a high level "
                "(e.g. 'explain the auth flow', 'how does the payment system work', 'what is the middleware doing'). "
                "Prefer this over search_codebase for 'explain' or 'how does X work' questions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Concept, feature, or system component to look up in documentation. E.g. 'auth middleware overview'"
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_schema",
            "description": (
                "Search the database schema index for table definitions, column names, and data types. "
                "Use ONLY when the user asks about database tables or columns "
                "(e.g. 'what does the users table store', 'what columns are in orders', 'what is the schema for payments'). "
                "Do not use for code or documentation questions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Table name or data domain to look up. E.g. 'users table' or 'payment transactions'"
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_diff",
            "description": (
                "Analyze the blast radius of a function signature change — find all files and callers that will break. "
                "Use when the user says they ARE CHANGING or HAVE CHANGED a function and want to know the impact "
                "(e.g. 'I changed X to require a new param', 'what breaks if I modify this function', 'blast radius of changing Y'). "
                "Requires the function name and the new or changed signature. "
                "Do NOT use for general code lookup — use search_codebase for that."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "changed_function": {
                        "type": "string",
                        "description": "The name of the function being changed. E.g. 'validate_token'"
                    },
                    "new_signature": {
                        "type": "string",
                        "description": "Description of the new signature or what changed. E.g. 'added scope parameter: validate_token(user_id, token, scope)'"
                    }
                },
                "required": ["changed_function", "new_signature"],
            },
        },
    },
]

In [35]:
routing_results_refactored = []  # collect for summary

for query, expected_tool in TEST_QUERIES:
    print(f"{'Query':<15} : {query}")
    print(f"{'Expected Tool':<15} : {expected_tool}")

    result = run_agent(query)
    actual_tools = result["tools_called"]
    first_tool = actual_tools[0] if actual_tools else "[no tool called]"

    routed_correctly = (first_tool == expected_tool)
    status = "✅ CORRECT" if routed_correctly else f"❌ WRONG — called {first_tool}"

    print(f"{'Tool called':<15}: {first_tool}  {status}")
    print(f"{'Answer':<15} : {result['answer']}")
    print("-" * 30)

    routing_results_refactored.append({
        "query": query,
        "expected": expected_tool,
        "actual": first_tool,
        "correct": routed_correctly,
        "answer": result["answer"],
    })

Query           : how does the auth middleware work?
Expected Tool   : search_docs
Tool called    : search_docs  ✅ CORRECT
Answer          : The auth middleware validates JWTs on every request [AUTH.md > Middleware].
------------------------------
Query           : what does the users table store?
Expected Tool   : search_schema
Tool called    : search_schema  ✅ CORRECT
Answer          : The users table stores the user ID, email, creation date, and role. [database > schema > users table]
------------------------------
Query           : I changed validate_token() to require a scope param — what files are affected?
Expected Tool   : analyze_diff
Tool called    : analyze_diff  ✅ CORRECT
Answer          : The files affected by the change to validate_token() are api/routes.py and tests/test_auth.py.
------------------------------
Query           : explain the payment flow
Expected Tool   : search_docs
Tool called    : search_docs  ✅ CORRECT
Answer          : The auth middleware validates JW